In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Timing experiments
import time

# Feature scaling and preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

# Pipeline construction
from sklearn.pipeline import Pipeline as SklearnPipeline
from imblearn.pipeline import Pipeline as ImbPipeline

# Class imbalance
from imblearn.over_sampling import SMOTE, RandomOverSampler

In [2]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression

# Random Forest
from sklearn.ensemble import RandomForestClassifier

# Gradient Boosting
from sklearn.ensemble import GradientBoostingClassifier

# XGBoost
from xgboost import XGBClassifier

# LightGBM
from lightgbm import LGBMClassifier

# CatBoost
from catboost import CatBoostClassifier

# Cross-validation
from sklearn.model_selection import StratifiedKFold, cross_validate

# Classification metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_curve,
    auc
)

# Hyperparameter tuning
from sklearn.model_selection import RandomizedSearchCV

# Model explainability
import shap

/home/emmizzy119/fraud-detection-system/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from pathlib import Path

# Directory containing the prepared data partitions
Split_Dir = Path("../data/processed")

# File paths
X_train_path = Split_Dir / "X_train.csv"
X_test_path = Split_Dir / "X_test.csv"
y_train_path = Split_Dir / "y_train.csv"
y_test_path = Split_Dir / "y_test.csv"

# Check whether all files exist
files ={
    "X_train": X_train_path,
    "X_test": X_test_path,
    "y_train": y_train_path,
    "y_test": y_test_path
}

for name, path in files.items():
    print(f"{name}: {'found' if path.exists() else 'not found'} - {path}")

X_train: found - ../data/processed/X_train.csv
X_test: found - ../data/processed/X_test.csv
y_train: found - ../data/processed/y_train.csv
y_test: found - ../data/processed/y_test.csv


In [5]:
X_train = pd.read_csv(X_train_path)
X_test = pd.read_csv(X_test_path)

y_train = pd.read_csv(y_train_path).squeeze("columns")
y_test = pd.read_csv(y_test_path).squeeze("columns")

print("Data loaded successfully.")

Data loaded successfully.


In [6]:
print("Training features:", X_train.shape)
print("Training target:", y_train.shape)

print("\nTest features:", X_test.shape)
print("Test target:", y_test.shape)

Training features: (226980, 30)
Training target: (226980,)

Test features: (56746, 30)
Test target: (56746,)


In [7]:
print("Training target classes:")
print(y_train.value_counts().sort_index())

print("\nTest target classes:")
print(y_test.value_counts().sort_index())

print("\nTraining target proportions:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nTest target proportions:")
print(y_test.value_counts(normalize=True).sort_index())

Training target classes:
Class
0    226602
1       378
Name: count, dtype: int64

Test target classes:
Class
0    56651
1       95
Name: count, dtype: int64

Training target proportions:
Class
0    0.998335
1    0.001665
Name: proportion, dtype: float64

Test target proportions:
Class
0    0.998326
1    0.001674
Name: proportion, dtype: float64


In [8]:
print("Same feature columns:", X_train.columns.equals(X_test.columns))
print("Training feature count:", X_train.shape[1])
print("Test feature count:", X_test.shape[1])

print("\nTraining columns:")
print(X_train.columns.tolist())

Same feature columns: True
Training feature count: 30
Test feature count: 30

Training columns:
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']


In [9]:
# Reproducibile experiment configuration
RANDOM_STATE = 42
N_SPLITS = 5

# Five-fold stratifired cross-validation
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print(f"Number of folds: {N_SPLITS}")
print(f"Random state: {RANDOM_STATE}")

Number of folds: 5
Random state: 42


In [10]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "average_precision": "average_precision",
    "roc_auc": "roc_auc"
}

print("Evaluation metrics:")
for metric in scoring:
    print(f"- {metric}")

Evaluation metrics:
- accuracy
- precision
- recall
- f1
- average_precision
- roc_auc


In [11]:
# Models to compare
MODEL_NAMES = [
    "Logistic Regression",
    "Random Forest",
    "Gradient Boosting",
    "XGBoost",
    "LightGBM",
    "CatBoost"
]

# Imbalance-handling strategies
STRATEGY_NAMES = [
    "Class Weighting",
    "SMOTE Oversampling",
    "Random Undersampling"
]

TOTAL_EXPERIMENTS = len(MODEL_NAMES) * len(STRATEGY_NAMES)

print(f"Number of models: {len(MODEL_NAMES)}")
print(f"Number of imbalance strategies: {len(STRATEGY_NAMES)}")
print(f"Total model-strategy combinations: {TOTAL_EXPERIMENTS}")
print(f"Total cross-validation fits: {TOTAL_EXPERIMENTS * N_SPLITS}")

Number of models: 6
Number of imbalance strategies: 3
Total model-strategy combinations: 18
Total cross-validation fits: 90


In [12]:
# 1. Logistic Regression
logistic_regression = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)

# 2. Random Forest
random_forest = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# 3. Gradient Boosting
gradient_boosting = GradientBoostingClassifier(
    random_state=RANDOM_STATE
)

# 4. XGBoost
xgboost_model = XGBClassifier(
    random_state=RANDOM_STATE,
    eval_metric="logloss",
    n_jobs=-1
)

# 5. LightGBM
lightgbm_model = LGBMClassifier(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)

# 6. CatBoost
catboost_model = CatBoostClassifier(
    random_seed=RANDOM_STATE,
    verbose=0,
    thread_count=-1
)

# Collect the models
models = {
    "Logistic Regression": logistic_regression,
    "Random Forest": random_forest,
    "Gradient Boosting": gradient_boosting,
    "XGBoost": xgboost_model,
    "LightGBM": lightgbm_model,
    "CatBoost": catboost_model
}

print(f"Models defined: {len(models)}")

for name in models:
    print(name)

Models defined: 6
Logistic Regression
Random Forest
Gradient Boosting
XGBoost
LightGBM
CatBoost


In [13]:
from imblearn.under_sampling import RandomUnderSampler

In [14]:
# Columns to scale for Logistic Regression
scale_columns = ["Time", "Amount"]

# All predictors are numerical
all_feature_columns = X_train.columns.tolist()

print("Number of features:", len(all_feature_columns))
print("Columns scaled for Logistic Regression:", scale_columns)
print("Features scaled before SMOTE:", len(all_feature_columns))

Number of features: 30
Columns scaled for Logistic Regression: ['Time', 'Amount']
Features scaled before SMOTE: 30


In [22]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler


def build_pipeline(model, strategy):
    """Build a model pipeline for the selected imbalance strategy."""

    model = clone(model)
    steps = []

    supported_models = (
        LogisticRegression,
        RandomForestClassifier,
        GradientBoostingClassifier,
        XGBClassifier,
        LGBMClassifier,
        CatBoostClassifier
    )

    if not isinstance(model, supported_models):
        raise TypeError(f"Unsupported model type: {type(model).__name__}")

    if strategy == "Class Weighting":

        if isinstance(model, LogisticRegression):
            steps.append((
                "scaler",
                ColumnTransformer(
                    transformers=[
                        ("scale", StandardScaler(), ["Time", "Amount"])
                    ],
                    remainder="passthrough"
                )
            ))

        if isinstance(model, (LogisticRegression, RandomForestClassifier)):
            model.set_params(class_weight="balanced")

        elif isinstance(model, XGBClassifier):
            model = ClassWeightedClassifier(model)

        elif isinstance(model, GradientBoostingClassifier):
            model = ClassWeightedClassifier(model)

        elif isinstance(model, LGBMClassifier):
            model.set_params(class_weight="balanced")

        elif isinstance(model, CatBoostClassifier):
            model.set_params(auto_class_weights="Balanced")

    elif strategy == "SMOTE Oversampling":
        steps.append(("scaler", StandardScaler()))
        steps.append(("sampler", SMOTE(random_state=RANDOM_STATE)))

    elif strategy == "Random Undersampling":

        if isinstance(model, LogisticRegression):
            steps.append((
                "scaler",
                ColumnTransformer(
                    transformers=[
                        ("scale", StandardScaler(), ["Time", "Amount"])
                    ],
                    remainder="passthrough"
                )
            ))

        steps.append((
            "sampler",
            RandomUnderSampler(random_state=RANDOM_STATE)
        ))

    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    steps.append(("classifier", model))

    return ImbPipeline(steps=steps)

In [21]:
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.utils.validation import check_is_fitted


class ClassWeightedClassifier(ClassifierMixin, BaseEstimator):

    def __init__(self, estimator):
        self.estimator = estimator

    def fit(self, X, y):
        self.estimator_ = clone(self.estimator)

        # Calculate weights using this fold's training labels.
        sample_weights = compute_sample_weight(
            class_weight="balanced",
            y=y
        )

        # XGBoost uses scale_pos_weight instead.
        if isinstance(self.estimator_, XGBClassifier):
            class_counts = np.bincount(np.asarray(y, dtype=int))
            self.estimator_.set_params(
                scale_pos_weight=class_counts[0] / class_counts[1]
            )
            self.estimator_.fit(X, y)

        # Gradient Boosting accepts sample_weight directly.
        elif isinstance(self.estimator_, GradientBoostingClassifier):
            self.estimator_.fit(
                X, y,
                sample_weight=sample_weights
            )

        else:
            raise TypeError(
                "This wrapper supports Gradient Boosting and XGBoost."
            )

        self.classes_ = self.estimator_.classes_
        return self

    def predict(self, X):
        check_is_fitted(self, "estimator_")
        return self.estimator_.predict(X)

    def predict_proba(self, X):
        check_is_fitted(self, "estimator_")
        return self.estimator_.predict_proba(X) 